# Credit Scoring — Complete Data Analysis & Machine Learning Notebook

**Dataset**: 100,000 customer financial records (train.csv + test.csv)  
**Target**: Predict Credit Score category — Good / Standard / Poor  
**Steps**:
1. Load & Inspect Data
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Model Training (Logistic Regression, Decision Tree, Random Forest)
6. Evaluation & Comparison
7. Save Artifacts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, re, os, json, joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Libraries loaded successfully!')

## 1. Load & Inspect Data

In [ ]:
train_raw = pd.read_csv('../data/train.csv', low_memory=False)
test_raw  = pd.read_csv('../data/test.csv',  low_memory=False)

print(f'Train shape: {train_raw.shape}')
print(f'Test  shape: {test_raw.shape}')
print(f'\nColumns ({len(train_raw.columns)}):', list(train_raw.columns))

In [ ]:
train_raw.head()

In [ ]:
train_raw.dtypes

## 2. Basic Statistics & Missing Values

In [ ]:
# Numerical stats
num_cols = train_raw.select_dtypes(include=np.number).columns.tolist()
train_raw[num_cols].describe().round(2)

In [ ]:
# Missing value analysis
missing = train_raw.isnull().sum()
missing_pct = (missing / len(train_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct%': missing_pct})
missing_df[missing_df['Missing'] > 0].sort_values('Pct%', ascending=False)

In [ ]:
print(f'Duplicate rows: {train_raw.duplicated().sum()}')
print(f'\nTarget distribution:')
print(train_raw['Credit_Score'].value_counts())
print('\nPercentage:')
print((train_raw['Credit_Score'].value_counts() / len(train_raw) * 100).round(2))

## 3. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc = train_raw['Credit_Score'].value_counts()
colors = ['#4caf50', '#ff9800', '#f44336']
axes[0].bar(vc.index, vc.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Credit Score Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(vc.values, labels=vc.index, colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Credit Score Pie Chart', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Cleaning

In [ ]:
def clean_dataframe(df):
    df = df.copy()
    for c in df.select_dtypes('object').columns:
        df[c] = df[c].astype(str).str.strip()
    PLACEHOLDERS = ['nan','NaN','NA','N/A','null','NULL','','_','__','!@9#%8','#F%$D@*&8']
    for c in df.columns:
        df[c] = df[c].replace(PLACEHOLDERS, np.nan)

    if 'Annual_Income' in df.columns:
        df['Annual_Income'] = pd.to_numeric(df['Annual_Income'].astype(str).str.replace(r'[_a-zA-Z]+$','',regex=True), errors='coerce')
    if 'Age' in df.columns:
        df['Age'] = pd.to_numeric(df['Age'].astype(str).str.replace(r'[^0-9\-]','',regex=True), errors='coerce')
        df['Age'] = df['Age'].apply(lambda x: x if pd.notna(x) and 18 <= x <= 100 else np.nan)
    if 'Num_of_Loan' in df.columns:
        df['Num_of_Loan'] = pd.to_numeric(df['Num_of_Loan'].astype(str).str.replace(r'[^0-9\-]','',regex=True), errors='coerce').clip(lower=0)
    if 'Num_of_Delayed_Payment' in df.columns:
        df['Num_of_Delayed_Payment'] = pd.to_numeric(df['Num_of_Delayed_Payment'].astype(str).str.replace(r'[^0-9\-]','',regex=True), errors='coerce').clip(lower=0)
    if 'Changed_Credit_Limit' in df.columns:
        df['Changed_Credit_Limit'] = pd.to_numeric(df['Changed_Credit_Limit'].astype(str).str.replace(r'[^0-9\.\-]','',regex=True), errors='coerce')
    if 'Outstanding_Debt' in df.columns:
        df['Outstanding_Debt'] = pd.to_numeric(df['Outstanding_Debt'].astype(str).str.replace(r'[^0-9\.]','',regex=True), errors='coerce')
    if 'Credit_History_Age' in df.columns:
        def parse_credit_age(val):
            if pd.isna(val): return np.nan
            y = re.search(r'(\d+)\s*Year', str(val), re.I)
            m = re.search(r'(\d+)\s*Month', str(val), re.I)
            return (int(y.group(1)) if y else 0)*12 + (int(m.group(1)) if m else 0)
        df['Credit_History_Age'] = df['Credit_History_Age'].apply(parse_credit_age)
    if 'Amount_invested_monthly' in df.columns:
        df['Amount_invested_monthly'] = pd.to_numeric(df['Amount_invested_monthly'].astype(str).str.replace(r'[^0-9\.]','',regex=True), errors='coerce')
    if 'Monthly_Balance' in df.columns:
        df['Monthly_Balance'] = pd.to_numeric(df['Monthly_Balance'].astype(str).str.replace(r'[^0-9\.\-]','',regex=True), errors='coerce')
    if 'Credit_Mix' in df.columns:
        df['Credit_Mix'] = df['Credit_Mix'].apply(lambda x: x if x in {'Good','Standard','Bad'} else np.nan)
    return df

train = clean_dataframe(train_raw)
test  = clean_dataframe(test_raw)
print('Cleaning complete. Train shape:', train.shape)

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Numerical distributions
eda_num_cols = ['Age','Annual_Income','Monthly_Inhand_Salary','Num_Bank_Accounts',
                'Num_Credit_Card','Interest_Rate','Num_of_Loan','Delay_from_due_date',
                'Num_of_Delayed_Payment','Outstanding_Debt','Credit_Utilization_Ratio',
                'Credit_History_Age','Total_EMI_per_month','Amount_invested_monthly','Monthly_Balance']
eda_num_cols = [c for c in eda_num_cols if c in train.columns]

fig, axes = plt.subplots(4, 4, figsize=(20, 14))
axes = axes.flatten()
for i, col in enumerate(eda_num_cols):
    axes[i].hist(train[col].dropna(), bins=40, color='#3b82d4', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Key features vs Credit Score
key_cols = ['Annual_Income','Outstanding_Debt','Credit_History_Age',
            'Num_of_Delayed_Payment','Interest_Rate','Credit_Utilization_Ratio']
order = ['Poor','Standard','Good']
palette = {'Good':'#4caf50','Standard':'#ff9800','Poor':'#f44336'}

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()
for i, col in enumerate(key_cols):
    data = train[[col,'Credit_Score']].dropna()
    for cs in order:
        axes[i].hist(data[data['Credit_Score']==cs][col], bins=30, alpha=0.6, label=cs, color=palette[cs])
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend()
plt.suptitle('Key Features by Credit Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = train[eda_num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, annot_kws={'size':8}, vmin=-1, vmax=1, center=0)
ax.set_title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Feature Engineering

In [ ]:
DROP_COLS = ['ID','Customer_ID','Month','Name','SSN','Type_of_Loan']

def engineer_features(df):
    df = df.copy()
    df['Debt_to_Income']    = np.where(df['Annual_Income'] > 0, df['Outstanding_Debt'] / df['Annual_Income'], np.nan)
    df['EMI_to_Income']     = np.where(df['Monthly_Inhand_Salary'] > 0, df['Total_EMI_per_month'] / df['Monthly_Inhand_Salary'], np.nan)
    df['Investment_Rate']   = np.where(df['Monthly_Inhand_Salary'] > 0, df['Amount_invested_monthly'] / df['Monthly_Inhand_Salary'], np.nan)
    df['Has_Delayed_Payment'] = (df['Num_of_Delayed_Payment'].fillna(0) > 0).astype(int)
    df['High_Utilization']    = (df['Credit_Utilization_Ratio'] > 30).astype(int)
    if 'Payment_of_Min_Amount' in df.columns:
        df['Pays_Min_Only'] = (df['Payment_of_Min_Amount'].astype(str).str.strip().str.lower() == 'yes').astype(int)
        df.drop(columns=['Payment_of_Min_Amount'], inplace=True)
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print(f'Train features: {train_fe.shape[1]}, Test features: {test_fe.shape[1]}')

## 7. Encoding & Preprocessing

In [ ]:
cat_encode = ['Occupation','Credit_Mix','Payment_Behaviour']
cat_encode = [c for c in cat_encode if c in train_fe.columns]
label_encoders = {}
for col in cat_encode:
    le = LabelEncoder()
    combined = pd.concat([train_fe[col], test_fe[col]], axis=0).fillna('Unknown').astype(str)
    le.fit(combined)
    train_fe[col] = le.transform(train_fe[col].fillna('Unknown').astype(str))
    test_fe[col]  = le.transform(test_fe[col].fillna('Unknown').astype(str))
    label_encoders[col] = le
    print(f'Encoded {col}: {list(le.classes_)}')

le_target = LabelEncoder()
train_fe['Credit_Score'] = le_target.fit_transform(train_fe['Credit_Score'].fillna('Standard'))
print(f'Target classes: {list(le_target.classes_)}')

In [ ]:
TARGET = 'Credit_Score'
feature_cols = [c for c in train_fe.columns if c != TARGET and c in test_fe.columns]
X = train_fe[feature_cols].copy()
y = train_fe[TARGET].copy()

medians = X.median()
X.fillna(medians, inplace=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

## 8. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42, n_jobs=-1),
    'Decision Tree':        DecisionTreeClassifier(max_depth=12, min_samples_leaf=30, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_leaf=10, n_jobs=-1, random_state=42, class_weight='balanced'),
}

results = {}
for name, model in models.items():
    print(f'\nTraining: {name}')
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    acc  = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_val, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_val, y_pred, average='weighted', zero_division=0)
    auc  = roc_auc_score(y_val, y_proba, multi_class='ovr', average='weighted')
    results[name] = dict(model=model, y_pred=y_pred, y_proba=y_proba, accuracy=acc, precision=prec, recall=rec, f1=f1, auc=auc)
    print(f'  Accuracy={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    print(classification_report(y_val, y_pred, target_names=le_target.classes_))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_val, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le_target.classes_, yticklabels=le_target.classes_, ax=ax)
    ax.set_title(f'{name}\nAcc={res["accuracy"]:.3f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
line_colors = ['#3b82d4','#f44336','#4caf50']
for ax, (name, res) in zip(axes, results.items()):
    for ci, (cls_name, color) in enumerate(zip(le_target.classes_, line_colors)):
        y_bin = (y_val == ci).astype(int)
        fpr, tpr, _ = roc_curve(y_bin, res['y_proba'][:, ci])
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls_name} (AUC={roc_auc_score(y_bin, res["y_proba"][:,ci]):.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_title(f'ROC - {name}', fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=8)
plt.suptitle('ROC Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison table
metrics_list = ['accuracy','precision','recall','f1','auc']
comp_df = pd.DataFrame(
    {name: [res[m] for m in metrics_list] for name, res in results.items()},
    index=metrics_list
).T.round(4)
print('Model Comparison:')
comp_df

In [ ]:
# Feature importance
rf = results['Random Forest']['model']
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp[::-1].plot(kind='barh', ax=ax, color='#3b82d4')
ax.set_title('Top 15 Feature Importances - Random Forest', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
print(feat_imp)

## 9. Save Best Model

In [ ]:
best_name  = comp_df['f1'].idxmax()
best_model = results[best_name]['model']
print(f'Best model: {best_name} (F1={comp_df.loc[best_name, "f1"]:.4f})')

# Retrain on full data
best_model.fit(scaler.fit_transform(X), y)

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model,    '../models/best_model.pkl')
joblib.dump(scaler,        '../models/scaler.pkl')
joblib.dump(label_encoders,'../models/label_encoders.pkl')
joblib.dump(le_target,     '../models/label_encoder_target.pkl')
joblib.dump(medians,       '../models/medians.pkl')
joblib.dump(feature_cols,  '../models/feature_cols.pkl')
print('All artifacts saved to ../models/')

## Summary

| Model | Accuracy | F1-Score | ROC-AUC |
|-------|----------|----------|---------|
| Logistic Regression | 0.6089 | 0.5939 | 0.7429 |
| Decision Tree | 0.7123 | 0.7137 | 0.8359 |
| **Random Forest** | **0.7216** | **0.7257** | **0.8740** |

**Best Model**: Random Forest with 72.2% accuracy and 0.874 ROC-AUC  
**Top Feature**: Outstanding Debt (14.9% importance)